# Sentiment / Emotion Classifier

Build a multi-class classifier using Recurrent Neural Networks or Transformers to classify the emotional tone of the customer's message, mapping to negative/neutral/positive.

In [8]:
import sys
!{sys.executable} -m pip install datasets torch transformers pandas scikit-learn joblib

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.3 MB 4.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.6/12.3 MB 3.0 MB/s eta 0:00:04
   ------- -------------------------------- 2.4/12.3 MB 3.3 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/12.3 MB 3.4 MB/s eta 0:00:03
   ----------- ---------------------------- 3.7/12.3 MB 3.4 MB/s eta 0:00:03
   -------------- ------------------------- 4.5/12.3 MB 3.4 MB/s eta 0:00:03
   ---------------- ----------------------- 5.0/12.3 MB 3.4 MB/s eta 0:00:03
   ------------------ --------------------- 5.8/12.3 MB 3.3 MB/s eta 0:00:02
   -------------------- ------------------- 6.3/12.3 MB 3.2 MB/s eta 0:00:02
   --------------------- ------------------ 6.6/12.3 MB 3.1 MB/s eta 0:00:02
   ----------------------- ---------------- 7.1/12.3 MB 3.0 MB/s eta 0:00:02
   ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## Load Dataset
Using `dair-ai/emotion`. We will map the 6 emotions to negative, neutral, positive.

In [9]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('dair-ai/emotion', 'split')
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])
test_df = pd.DataFrame(dataset['test'])

# Emotion mapping: 
# 0: sadness (negative), 1: joy (positive), 2: love (positive), 3: anger (negative), 4: fear (negative), 5: surprise (neutral)
emotion_map = {0: 'negative', 1: 'positive', 2: 'positive', 3: 'negative', 4: 'negative', 5: 'neutral'}
train_df['sentiment'] = train_df['label'].map(emotion_map)
val_df['sentiment'] = val_df['label'].map(emotion_map)
test_df['sentiment'] = test_df['label'].map(emotion_map)
train_df.head()

,text,label,sentiment
0,i didnt feel humiliated,0,negative
1,i can go from feeling so hopeless to so damned...,0,negative
2,im grabbing a minute to post i feel greedy wrong,3,negative
3,i am ever feeling nostalgic about the fireplac...,2,positive
4,i am feeling grouchy,3,negative


## Model Preparation
We use a pre-trained small transformer (DistilBERT).

In [10]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_df['target'] = le.fit_transform(train_df['sentiment'])
val_df['target'] = le.transform(val_df['sentiment'])
test_df['target'] = le.transform(test_df['sentiment'])

model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128)
        self.labels = labels.tolist()
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_loader = DataLoader(EmotionDataset(train_df['text'], train_df['target']), batch_size=32, shuffle=True)
val_loader = DataLoader(EmotionDataset(val_df['text'], val_df['target']), batch_size=32)
test_loader = DataLoader(EmotionDataset(test_df['text'], test_df['target']), batch_size=32)

c:\Users\LEGION\anaconda3\envs\dental_xray_env\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LEGION\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## Training Loop

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(1): # Adjust epochs as needed
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device), labels=batch['labels'].to(device))
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    print(f'Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}')

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1591.80it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 Loss: 0.1625


## Evaluation

In [12]:
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

model.eval()
preds, true_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        true_labels.extend(batch['labels'].numpy())

print('Test Accuracy:', accuracy_score(true_labels, preds))
print(classification_report(true_labels, preds, target_names=le.classes_))

Test Accuracy: 0.9735
              precision    recall  f1-score   support

    negative       0.99      0.97      0.98      1080
     neutral       0.70      0.82      0.76        66
    positive       0.98      0.99      0.98       854

    accuracy                           0.97      2000
   macro avg       0.89      0.93      0.91      2000
weighted avg       0.98      0.97      0.97      2000



## Save Model

In [13]:
import os
import joblib
os.makedirs('models/sentiment_model', exist_ok=True)
model.save_pretrained('models/sentiment_model')
tokenizer.save_pretrained('models/sentiment_model')
joblib.dump(le, 'models/sentiment_encoder.pkl')
print('Model saved')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.97it/s]

Model saved
